## Pricing Comparison

In [4]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

DATA_PATH = Path("../data/processed/pricing_analyst_cleaned.csv")
FIGURES_DIR = Path("../outputs/figures")
TABLES_DIR = Path("../outputs/tables")
TABLE_PATH1 = Path("../outputs/tables/summary_table.csv") 
TABLE_PATH2 = Path("../outputs/tables/revenue_realisation_by_strategy.csv") 
TABLE_PATH3 = Path("../outputs/tables/statistical_tests.csv")

df_analysis = pd.read_csv(DATA_PATH)
summary_table = pd.read_csv(TABLE_PATH1)
revenue_realisation = pd.read_csv(TABLE_PATH2)
statistical_tests = pd.read_csv(TABLE_PATH3)

date_columns = ["offerdate", "purchase_date"]
for col in date_columns:
    df_analysis[col] = pd.to_datetime(df_analysis[col], errors="coerce")

print(df_analysis.shape)
df_analysis.info()
display(df_analysis.head())

(8865, 24)
<class 'pandas.DataFrame'>
RangeIndex: 8865 entries, 0 to 8864
Data columns (total 24 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   offerdate                      8865 non-null   datetime64[us]
 1   sold_premium                   1991 non-null   float64       
 2   offered_premium                8865 non-null   float64       
 3   purchase_price                 8864 non-null   float64       
 4   purchase_date                  8865 non-null   datetime64[us]
 5   item_age                       8862 non-null   float64       
 6   pricing_point                  8865 non-null   str           
 7   predictedconversionrate        8865 non-null   float64       
 8   plan_flag                      8865 non-null   int64         
 9   plan_count                     8824 non-null   float64       
 10  plansactive_lastyear_count     8824 non-null   float64       
 11  planscancelled_la

,offerdate,sold_premium,offered_premium,purchase_price,purchase_date,item_age,pricing_point,predictedconversionrate,plan_flag,plan_count,...,price_diff,ismodel,sale_flag,base_rate,manufacturerbrandname_enc,itemcategoryname_enc,itemsupercategorycode_enc,invalid_price_flag,price_diff_check,price_diff_matches
0,2023-03-17,NaN,32.64,89.99,2023-03-16,1.0,@22%,0.15,0,0.0,...,-0.058824,yes,0,34.68,56,35,14,False,-1.981176,False
1,2023-03-01,69.72,69.72,329.00,2022-12-24,67.0,@22%,0.81,1,5.0,...,-0.023529,yes,1,71.40,123,34,3,False,-1.656471,False
2,2023-04-12,NaN,48.24,249.00,2023-04-05,7.0,@23%,0.08,0,0.0,...,0.210843,yes,0,39.84,7,16,12,False,8.189157,False
3,2023-03-09,NaN,91.92,746.42,2021-03-09,730.0,@23%,0.32,0,0.0,...,0.298305,yes,0,70.80,107,36,4,False,20.821695,False
4,2023-03-18,NaN,89.64,493.98,2023-03-18,0.0,@22%,0.25,1,1.0,...,0.299130,yes,0,69.00,57,36,4,False,20.340870,False


In [23]:
task_d_overview = summary_table.copy()

task_d_overview["avg_realised_uplift"] = revenue_realisation["avg_realised_uplift"]
task_d_overview["total_realised_uplift"] = revenue_realisation["total_realised_uplift"]

display(task_d_overview)

task_d_overview = task_d_overview.set_index("pricing_point")
task_d_overview.to_csv("../outputs/tables/task_d_overview.csv", index=False)

,pricing_point,no_of_offers,avg_base_rate,avg_offered_premium,avg_sold_premium,avg_price_increase,conversion_rate,conversion_rate_pct,avg_realised_uplift,total_realised_uplift
0,ASIS FEE,1115,48.973668,48.973668,49.572923,0.000000,0.233184,23.32,0.000000,0.00
1,@22%,3923,49.200958,55.212603,55.750694,0.124322,0.220240,22.02,6.417917,5545.08
2,@23%,3827,49.014236,51.976650,51.482768,0.059686,0.226548,22.65,2.923599,2534.76


In [6]:
def get_pvalue(comparison_label):
    return statistical_tests.loc[
        statistical_tests["comparison"] == comparison_label, "p_value"
    ].values[0]

significance_summary = pd.DataFrame({
    "comparison": [
        "ASIS vs @22% (offered_premium)",
        "ASIS vs @23% (offered_premium)",
        "@22% vs @23% (offered_premium)",
        "@22% vs @23% (price_diff)",
        "All strategies (conversion, chi-square)"
    ]
})

significance_summary["p_value"] = significance_summary["comparison"].apply(get_pvalue)
significance_summary["significant_at_5pct"] = significance_summary["p_value"] < 0.05

display(significance_summary)
significance_summary.to_csv("../outputs/tables/task_d_significance_summary.csv", index=False)

,comparison,p_value,significant_at_5pct
0,ASIS vs @22% (offered_premium),3.465742e-36,True
1,ASIS vs @23% (offered_premium),8.160815e-10,True
2,@22% vs @23% (offered_premium),5.815867e-18,True
3,@22% vs @23% (price_diff),1.515599e-101,True
4,"All strategies (conversion, chi-square)",6.115687e-01,False


In [19]:
ranking_table = pd.DataFrame({
    "criterion": [
        "Statistical: conversion difference vs ASIS",
        "Statistical: premium difference vs ASIS",
        "Revenue: avg realised uplift per sale",
        "Revenue: total realised uplift"
    ],
    "best_strategy": [
        task_d_overview["conversion_rate"].drop("ASIS FEE").idxmax(),
        task_d_overview["avg_offered_premium"].drop("ASIS FEE").idxmax(),
        task_d_overview["avg_realised_uplift"].idxmax(),
        task_d_overview["total_realised_uplift"].idxmax()
    ],
    "value": [
        task_d_overview["conversion_rate"].drop("ASIS FEE").max(),
        task_d_overview["avg_offered_premium"].drop("ASIS FEE").max(),
        task_d_overview["avg_realised_uplift"].max(),
        task_d_overview["total_realised_uplift"].max()
    ]
})

display(ranking_table)
ranking_table.to_csv("../outputs/tables/task_d_ranking.csv", index=False)

,criterion,best_strategy,value
0,Statistical: conversion difference vs ASIS,@23%,0.226548
1,Statistical: premium difference vs ASIS,@22%,55.212603
2,Revenue: avg realised uplift per sale,@22%,6.417917
3,Revenue: total realised uplift,@22%,5545.080000


In [24]:
print("=== Task D: Strategy Performance Overview ===\n")
display(task_d_overview)

print("\n=== Statistical Significance ===\n")
display(significance_summary)

print("\n=== Best Strategy by Criterion ===\n")
display(ranking_table)

=== Task D: Strategy Performance Overview ===



,no_of_offers,avg_base_rate,avg_offered_premium,avg_sold_premium,avg_price_increase,conversion_rate,conversion_rate_pct,avg_realised_uplift,total_realised_uplift
pricing_point,,,,,,,,,
ASIS FEE,1115,48.973668,48.973668,49.572923,0.000000,0.233184,23.32,0.000000,0.00
@22%,3923,49.200958,55.212603,55.750694,0.124322,0.220240,22.02,6.417917,5545.08
@23%,3827,49.014236,51.976650,51.482768,0.059686,0.226548,22.65,2.923599,2534.76



=== Statistical Significance ===



,comparison,p_value,significant_at_5pct
0,ASIS vs @22% (offered_premium),3.465742e-36,True
1,ASIS vs @23% (offered_premium),8.160815e-10,True
2,@22% vs @23% (offered_premium),5.815867e-18,True
3,@22% vs @23% (price_diff),1.515599e-101,True
4,"All strategies (conversion, chi-square)",6.115687e-01,False



=== Best Strategy by Criterion ===



,criterion,best_strategy,value
0,Statistical: conversion difference vs ASIS,@23%,0.226548
1,Statistical: premium difference vs ASIS,@22%,55.212603
2,Revenue: avg realised uplift per sale,@22%,6.417917
3,Revenue: total realised uplift,@22%,5545.080000
